In [ ]:
# PhysioNet MI source-only LOSO | Kaggle job 7/10 | targets S67-S77
# Settings: Accelerator = GPU T4, Internet = On. Run with "Save Version -> Save & Run All".
# Data: dataset dmc2003vn/physionet-mi-106subj (attach via Add Input, or it is downloaded from the public API).
import codecs, os, select, shutil, subprocess, sys, time
from pathlib import Path

BRANCH = 'feature/full-mamba-source-only-physionet109'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
TARGET_SUBJECTS = list(range(67, 77 + 1))
EXCLUDED = {88, 92, 100}   # must match exclude_subject_ids in the config
BATCH_SIZE = 96
JOB_TAG = 's067-s077'
DATASET = 'dmc2003vn/physionet-mi-106subj'      # public Kaggle Dataset with the decoded epochs
CACHE_FILE = 'physionet_106subj_140d39fd97.npz'  # name must match the config hash

REPO_PATH = Path('/kaggle/working/repo')
MNE_DATA = Path('/tmp/mne_data')            # EDF downloads (kept out of the output)
ARRAY_CACHE = Path('/tmp/physionet_arrays')  # decoded epochs shared by every fold
OUTPUT = Path(f'/kaggle/working/physionet109_source_only_{JOB_TAG}')
OUTPUT.mkdir(parents=True, exist_ok=True)


def run(command, cwd=None, env=None, log_path=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, env=env, check=True)
        return
    process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, bufsize=0)
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    started = time.monotonic()
    with open(log_path, 'w', encoding='utf-8') as log:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                text = decoder.decode(chunk)
                print(text, end='', flush=True); log.write(text); log.flush()
            elif process.poll() is None:
                beat = f'[heartbeat] elapsed={(time.monotonic() - started) / 60:.1f}m\n'
                print(beat, end='', flush=True); log.write(beat); log.flush()
            else:
                break
    if process.wait() != 0:
        raise RuntimeError(f'Command failed. Log: {log_path}')


# ---- Setup ----
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)
UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1',
     '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)
run(['nvidia-smi'])

environment = os.environ.copy()
environment.update({'PYTHONUNBUFFERED': '1', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
                    'MPLBACKEND': 'Agg', 'MNE_DATA': str(MNE_DATA),
                    'MNE_DATASETS_EEGBCI_PATH': str(MNE_DATA)})
MNE_DATA.mkdir(parents=True, exist_ok=True)

# Reuse the decoded epochs from the public Kaggle Dataset (made by job 0):
# attached via "Add Input" if present, otherwise downloaded from the public API
# (kagglehub cannot attach a new dataset inside a "Save Version" run).
ARRAY_CACHE.mkdir(parents=True, exist_ok=True)
cache_files = list(Path('/kaggle/input').rglob(CACHE_FILE))
if not cache_files:
    archive = Path('/tmp/physionet_dataset.zip')
    run(['curl', '-fL', '--retry', '5', '-o', archive,
         f'https://www.kaggle.com/api/v1/datasets/download/{DATASET}'])
    run(['unzip', '-o', '-q', archive, '-d', '/tmp/physionet_dataset'])
    archive.unlink()
    cache_files = list(Path('/tmp/physionet_dataset').rglob(CACHE_FILE))
if not cache_files:
    raise FileNotFoundError(f'{CACHE_FILE} not found in {DATASET}')
link = ARRAY_CACHE / CACHE_FILE
if not link.exists():
    link.symlink_to(cache_files[0])
print('Using PhysioNet cache:', cache_files[0], flush=True)

# ---- One process per fold; results are archived right after each fold ----
for subject in TARGET_SUBJECTS:
    if subject in EXCLUDED:
        print(f'Skip S{subject:03d}: excluded recording.', flush=True)
        continue
    if (OUTPUT / f'S{subject:03d}_results.zip').exists():
        print(f'Skip S{subject:03d}: already finished.', flush=True)
        continue
    override = (
        "import yaml; from pathlib import Path; "
        "p=Path('configs/full_mamba_source_only.yaml'); c=yaml.safe_load(p.read_text()); "
        f"c['subject_ids']=[{subject}]; c['log_every_n_epochs']=10; c['enable_progress_bar']=False; "
        f"c['preprocessing']['physionet']['batch_size']={BATCH_SIZE}; "
        "c['preprocessing']['physionet']['accumulate_grad_batches']=1; "
        f"c['preprocessing']['physionet']['array_cache_dir']='{ARRAY_CACHE}'; "
        "p.write_text(yaml.safe_dump(c, sort_keys=False))"
    )
    run([PYTHON, '-c', override], cwd=REPO_PATH)
    results_dir = REPO_PATH / 'results'
    shutil.rmtree(results_dir, ignore_errors=True)
    print(f'===== PhysioNet source-only | target S{subject:03d} | BS={BATCH_SIZE} =====', flush=True)
    run([PYTHON, '-u', 'train_pipeline.py', '--model', 'full_mamba_source_only',
         '--dataset', 'physionet', '--loso', '--gpu_id', '0'],
        cwd=REPO_PATH, env=environment, log_path=OUTPUT / f'S{subject:03d}.log')
    shutil.make_archive(str(OUTPUT / f'S{subject:03d}_results'), 'zip',
                        root_dir=REPO_PATH, base_dir='results')
    print(f'Saved {OUTPUT / f"S{subject:03d}_results.zip"}', flush=True)

# ---- Summary of this job ----
import json, zipfile
rows = []
for archive in sorted(OUTPUT.glob('S*_results.zip')):
    with zipfile.ZipFile(archive) as zf:
        for name in zf.namelist():
            if '/subjects/subject_' in name and name.endswith('.json'):
                rows.append(json.loads(zf.read(name)))
with open(OUTPUT / 'summary.csv', 'w') as f:
    f.write('subject_id,test_acc,test_kappa,best_val_acc,train_time_min\n')
    for r in sorted(rows, key=lambda r: r['subject_id']):
        f.write(f"{r['subject_id']},{r['test_acc']:.4f},{r['test_kappa']:.4f},"
                f"{r['best_val_acc']:.4f},{r['train_time_min']:.1f}\n")
print(open(OUTPUT / 'summary.csv').read())
